### Introduction

- Section 1:
    - Create 'k' different single path DLGN models, train each model on 'n' different depth 4, 100 dimensional ODT datasets and store the trained models. 
    - Each model/ dataset differs by their initialization seed. By default k=2000, n=5.
    - Configurations to replicate the results in Section 4.3 (in the paper):
        - Dataset 1: Data seed = 1009, Number of points = 1000000
        - Dataset 2: Data seed = 6755, Number of points = 1000000
        - Dataset 3: Data seed = 410455, Number of points = 1000000
        - Dataset 4: Data seed = 4287, Number of points = 2000000
        - Dataset 5: Data seed = 1108, Number of points = 2000000
        - Model seeds = [2, 4, 6, 8, ... 3998, 4000]
- Section 2:
    - Find out the models that converged to a low training loss (< 0.65) and to the same solution.
    - Compute the heuristic score for those models.

### 1. Train 'k' models on 'n' datasets

#### Imports

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import torch
import torch.nn as nn

from itertools import permutations
np.set_printoptions(precision=2)

from train import train_dlgn_2
from utils import set_torchseed
from dlgn import DLGN_SP
from odt import data_gen_decision_tree

#### Hyperparameters

In [ ]:
config = {
    'model_seed' : [i for i in range(2, 4002, 2)],
    # 'model_seed' : [2],
    'data_seed' : [1108, 4287, 1009, 410455, 6755],
    'input_dim' : 100,
    'output_dim' : 1,
    'hidden_layers' : 4,
    'hidden_neurons' : 1,
    'beta' : 100,
    'lr' : 0.2,
    'step': 2000,
    'optimizer' : 'SGD',
    'epochs' : 1000,
    'num_batches' : 1,
    'device' : 'cuda' if torch.cuda.is_available() else 'cpu',
    'gdecay' : 0,
    'vdecay' : 0,
    'mult': 1e-3
}

run_specs = {
    'save_models' : True, #Flag to decide whether to save the trained models
    'save_path' : f"models", #Folder where models will get stored
    'print' : 1000 #Interval at which statistics are to be printed
}

config['hidden_dims'] = [config['hidden_neurons']] * config['hidden_layers']

#### Create and Train Models

Note: The following cell trains 2000 different initializations of the single path DLGN on 5 different dataset. This process takes a considerable amount of time. To avoid this step and directly use the 368 initializations to run the heuristic solution predictor run the cells from Section 2.2.

In [ ]:
model = DLGN_SP(input_dim=config["input_dim"], 
                   output_dim=config["output_dim"], 
                   hidden_dims=config["hidden_dims"], 
                   beta=config["beta"], 
                   device=config["device"]).to(config["device"])

In [ ]:
all_models = {}  # Dict containing len(config['model_seed']) * len(config['data_seed']) models
# The key is a string f'{data_seed}_{model_seed}'. Each entry is inturn a dict containing the model state_dicts
# at initialization, at end of training and the epoch at which training loss decreased below 0.67.

for data_seed in config['data_seed']:

    print("-----------------------")
    print(f"Data seed = {data_seed}")

    set_torchseed(data_seed)

    n_samples = 1000000 if data_seed in [1009, 6755, 410455] else 2000000
    dim = 100
    num_levels = 4
    (X, y), (w_list, b_list, vals), stats = data_gen_decision_tree(
                                            dim=100, num_levels=4,
                                            num_data=n_samples)

    X = torch.tensor(X, dtype=torch.float)
    y = torch.tensor(y, dtype=torch.long).squeeze()
    w_list = torch.tensor(w_list, dtype=torch.float)
    b_list = torch.tensor(b_list, dtype=torch.float)
    vals = torch.tensor(vals, dtype=torch.float)

    num_data = len(X)
    num_train= num_data//2
    num_val = num_data//4
    num_test = num_data//4
    train_data = X[:num_train,:]
    train_data_labels = y[:num_train]

    val_data = X[num_train:num_train+num_val,:]
    val_data_labels = y[num_train:num_train+num_val]

    test_data = X[num_train+num_val :,:]
    test_data_labels = y[num_train+num_val :]

    
    for model_seed in config['model_seed']:

        print(f"Model Seed = {model_seed}")

        set_torchseed(model_seed)

        DLGN_init = DLGN_SP(input_dim=config["input_dim"], 
                   output_dim=config["output_dim"], 
                   hidden_dims=config["hidden_dims"], 
                   beta=config["beta"], 
                   device=config["device"]).to(config["device"])
        
        DLGN_init.value_layers.data=torch.ones_like(DLGN_init.value_layers.data)*2.0

        G0 = DLGN_init.gating_layers[0].weight.data
        G1 = DLGN_init.gating_layers[1].weight.data
        G2 = DLGN_init.gating_layers[2].weight.data
        G3 = DLGN_init.gating_layers[3].weight.data

        G0.copy_((torch.randint(2,(1,config['input_dim']))*2 -1.)*1.)
        G1.copy_((torch.randint(2,(1,config['input_dim']))*2 -1.)*1.)
        G2.copy_((torch.randint(2,(1,config['input_dim']))*2 -1.)*1.)
        G3.copy_((torch.randint(2,(1,config['input_dim']))*2 -1.)*1.)

        G0 *= config['mult']
        G1 *= config['mult']
        G2 *= config['mult']
        G3 *= config['mult']

        for name, param in DLGN_init.named_parameters():
            if 'val' in name:
                param.requires_grad_(False)

        DLGN_obj_store, mid_epoch, converged = train_dlgn_2(train_data=train_data,
                                            vali_data=val_data,
                                            test_data=test_data,
                                            train_labels=train_data_labels,
                                            vali_labels=val_data_labels,
                                            test_labels=test_data_labels,
                                            DLGN_obj=deepcopy(DLGN_init),
                                            config=config,
                                            run_specs=run_specs,
                                            plot=True,
                                            parameter_mask=None,
                                            use_wandb=False
                                            )
    
        if run_specs['save_models']:
            # os.makedirs(run_specs['save_path'], exist_ok=True)
            # save_path = os.path.join(run_specs['save_path'], f"data_{data_seed}_model_{model_seed}.pth")

            save_dict = {
                "init_dict" : DLGN_obj_store[0].state_dict(),
                "trained_dict" : DLGN_obj_store[-1].state_dict(),
                "mid_epoch" : DLGN_obj_store[mid_epoch+1].state_dict(),
                "converged" : converged #The converged flag tells us whether the train loss is below 0.65 for the model
            }

            # torch.save(save_dict, save_path)
        
        all_models[f'{data_seed}_{model_seed}'] = save_dict

        #Test
        with torch.no_grad():
            device = config['device']
            criterion = nn.CrossEntropyLoss(reduction="mean")
            inputs = test_data
            targets = test_data_labels
            inputs = inputs.to(device)
            targets = targets.to(device)
            preds = DLGN_obj_store[-1](inputs).reshape(-1,1)
            outputs = torch.cat((-1*preds, preds), dim=1)
            loss = criterion(outputs, targets)
            print("Test loss is: ",loss)
            print(" ")
        

In [ ]:
DLGN_init.gating_layers[0].weight[:, :15]

In [ ]:
os.makedirs(run_specs['save_path'], exist_ok=True)
save_path = os.path.join(run_specs['save_path'], f"all_models.pth")
torch.save(all_models, save_path)

### 2. Heuristic Solution Predictor

#### 2.1 Find runs that converged and to the same solution

In [ ]:
DLGN_init= DLGN_SP(input_dim=config["input_dim"], 
                   output_dim=config["output_dim"], 
                   hidden_dims=config["hidden_dims"], 
                   beta=config["beta"], 
                   device=device).to(config['device'])
DLGN_init.value_layers.data=torch.ones_like(DLGN_init.value_layers.data)*2.0

golden_seeds = [] #List of all models seeds that converged and to the same solution under all 5 datasets
for model_seed in config['model_seed']:

    all_models = torch.load("models/all_models.pth")

    sd1 = all_models[f"{config['data_seed'][0]}_{model_seed}"]
    sd2 = all_models[f"{config['data_seed'][1]}_{model_seed}"]
    sd3 = all_models[f"{config['data_seed'][2]}_{model_seed}"]
    sd4 = all_models[f"{config['data_seed'][3]}_{model_seed}"]
    sd5 = all_models[f"{config['data_seed'][4]}_{model_seed}"]

    if 'N' in (sd1['converged'], sd2['converged'], sd3['converged'], sd4['converged'], sd5['converged']):
        # at least 1 model did not converge. So skipping this seed.
        continue

    model1 = deepcopy(DLGN_init)
    model2 = deepcopy(DLGN_init)
    model3 = deepcopy(DLGN_init)
    model4 = deepcopy(DLGN_init)
    model5 = deepcopy(DLGN_init)

    model1.load_state_dict(sd1['trained_dict'])
    model2.load_state_dict(sd2['trained_dict'])
    model3.load_state_dict(sd3['trained_dict'])
    model4.load_state_dict(sd4['trained_dict'])
    model5.load_state_dict(sd5['trained_dict'])

    ew1 = model1.return_gating_functions()
    ew2 = model2.return_gating_functions()
    ew3 = model3.return_gating_functions()
    ew4 = model4.return_gating_functions()
    ew5 = model5.return_gating_functions()

    argmax1 = []
    argmax2 = []
    argmax3 = []
    argmax4 = []
    argmax5 = []

    for n in range(4):
        argmax1.append(torch.argmax(torch.abs(ew1[n])).item())
        argmax2.append(torch.argmax(torch.abs(ew2[n])).item())
        argmax3.append(torch.argmax(torch.abs(ew3[n])).item())
        argmax4.append(torch.argmax(torch.abs(ew4[n])).item())
        argmax5.append(torch.argmax(torch.abs(ew5[n])).item())

    if argmax1 == argmax2 == argmax3 == argmax4 == argmax5:
        # All 5 models have the same solution. Record this seed.
        golden_seeds.append(model_seed)

In [ ]:
len(golden_seeds) #This is the number of initializations that converged to the same solution for the 5 different datasets

#### 2.2 Load the 368 initializations

To avoid the expensive training process, the 368 initializations that converged to the same solution on the 5 different datasets are provided in the "load_368.pth" file. Un-comment the following cell to preload those initializations.

In [ ]:
import pickle

with open(r"./models/golden_seeds.pkl", "rb") as f:
    golden_seeds = pickle.load(f)

all_models = torch.load(r"./models/load_368.pth", map_location=config['device'])

#### 2.3 Compute Score

##### Util Functions

In [ ]:
def parent(i):
    if i==0:
        return -1
    else:
        return (i-1)//2


def level(i):
    ans= []
    for el in i:
        ans.append(int(np.log2(el+1)))
    return np.array(ans)

def give_paths_routes():
    paths = []
    routes = []
    for i in range(15,31):
        path= [i]
        route =[]
        for level in range(4):
            curr = path[-1]
            par = parent(curr)
            path.append(par)
            if curr == 2*par+1: # curr is left child of par
                route.append(-1)
            else:
                route.append(+1)
        paths.append(list(reversed(path)))
        routes.append(list(reversed(route)))
    paths = np.array(paths)
    routes = np.array(routes)
    return paths, routes    


In [ ]:
paths, routes = give_paths_routes()

##### Score Function

In [ ]:
def support_score(gating_weights, layer, tree_node):
    # e.g. say l=0, tree_node = 7
    w = np.array(gating_weights) # layers * dimension
    l = layer # 0 or 1 or 2 or 3
    i = tree_node # 0,1, ..., 14
    num_layers = 4
    lens_influence = 0
    paths, routes = give_paths_routes()
    for pi in range(len(paths)):
        cur_path = paths[pi][:-1] # removing the leaf node here
        leaf_node_id = paths[pi][-1]
        cur_route = routes[pi]
        if tree_node not in cur_path: # exclude all paths that don't contain tree_node
            continue
        target_label = 2*(leaf_node_id%2) -1 
        path_permuts = list(permutations(cur_path))
        route_permuts = list(permutations(cur_route))
        soln_mode_influence = 0
        best_mode = -1, -1

        for path_permut, route_permut in zip(path_permuts, route_permuts):
            curr_soln_mode_influence = 0 
            if path_permut[l] != tree_node: # e.g. only consider permutations of the path, that have 7 in position 0
                continue
            for l_dash in range(4): 
                if l== l_dash:
                    continue
                curr_soln_mode_influence += w[l_dash,path_permut[l_dash]]*route_permut[l_dash] # how does w_{l_dash} suport w_l to move along u_treenode
            if curr_soln_mode_influence > soln_mode_influence:
                soln_mode_influence = curr_soln_mode_influence
                best_mode = path_permut, route_permut
        
        soln_mode_influence *= cur_route[cur_path == tree_node][0]
        if target_label == -1:
            soln_mode_influence *= -1
        # print(cur_path, "\t", best_mode[0], "\t", best_mode[1], "  \t", leaf_node_id, "\t", target_label, "\t", soln_mode_influence)                
        lens_influence += soln_mode_influence
    
    self_influence = 0.
    for l_dash in range(4):    
        if l == l_dash:
            continue
        self_influence -= w[l_dash, tree_node] 

    # label_influence = 0.
    # if tree_node > 6:
    #     label_influence = -1 # all nodes want to move along -u_7 , - u_8, ..., -u_14 as they have a weak but global correlation with the class label
    # print("=================")
    # print(answer, "\t", self_influence, "\t", label_influence)
    # answer = np.array([lens_influence, self_influence, label_influence, gating_weights[layer,tree_node]]) 
    answer = np.array([lens_influence, self_influence, gating_weights[layer,tree_node]]) 
    #good coeffs = [0.5,2,1,1], 
    return answer    

In [ ]:
golden_weights = np.zeros((len(golden_seeds), 2, config['hidden_layers'], config['hidden_neurons'], config['input_dim']), dtype=np.float32)
# Runs , epochs, layers, neurons, dimension
DLGN_init = deepcopy(model)
for i, g in enumerate(golden_seeds):
    init = deepcopy(DLGN_init)
    # mid = deepcopy(DLGN_init)
    trained = deepcopy(DLGN_init)

    sd = all_models[f"{config['data_seed'][0]}_{g}"]
    
    init.load_state_dict(sd['init_dict'])
    # mid.load_state_dict(sd['mid_epoch'])
    trained.load_state_dict(sd['trained_dict'])

    ewi = init.return_gating_functions()
    # ewm = mid.return_gating_functions()
    ewt = trained.return_gating_functions()

    # ew = [ewi, ewm, ewt]
    ew = [ewi, ewt]
    temp = torch.stack([torch.stack(inner_list, dim=0) for inner_list in ew], dim=0)

    golden_weights[i] = temp.cpu().numpy()

In [ ]:
support_scores_all_good_runs = np.zeros((len(golden_seeds), 4, 15, 3))
for i,run in enumerate(golden_seeds):
    ew = golden_weights[i, 0,:,0,:]
    support_scores = np.zeros((4,15,3)) # layers, dimension, 3 components of influence
    for layer in range(4):
        for tree_node in range(15):
            ss = support_score(ew, layer, tree_node)
            support_scores[layer,tree_node] = ss
        #     print(layer,"\t", tree_node, "\t", ss)
        # print("===============")
    support_scores_all_good_runs[i] = support_scores


In [ ]:
path_scores_all_good_runs = []
for i,run in enumerate(golden_seeds):
    support_scores = np.array(support_scores_all_good_runs[i])
    paths, routes = give_paths_routes()
    path_scores = dict()
    for pi in range(len(paths)):
        cur_path = paths[pi][:-1] # removing the leaf node here
        leaf_node_id = paths[pi][-1]
        cur_route = routes[pi]
        path_permuts = list(permutations(cur_path))
        route_permuts = list(permutations(cur_route))
        for path_permut, route_permut in zip(path_permuts, route_permuts):
            ord_path_score = 0.
            for l_dash in range(4):
                ord_path_score += support_scores[l_dash, path_permut[l_dash]]*route_permut[l_dash]
            path_scores[tuple(path_permut+route_permut)] = ord_path_score
    path_scores_all_good_runs.append(path_scores)



In [ ]:
dual_path_scores_all_good_runs = []
for i,run in enumerate(golden_seeds):
    dual_path_scores = dict()
    path_scores = path_scores_all_good_runs[i]
    for k in path_scores.keys():
        dual_leaf_path = np.array(k)
        leaf_node =  dual_leaf_path[dual_leaf_path>6][0]
        where_leaf = np.where(dual_leaf_path==leaf_node)[0][0]
        par = (leaf_node-1)//2
        if leaf_node == 2*par+1:
            other_leaf_node = leaf_node+1
        else:
            other_leaf_node = leaf_node-1
        where_par = np.where(dual_leaf_path==par)[0][0]
        dual_leaf_path[where_par+4] = dual_leaf_path[where_leaf+4]
        dual_leaf_path[where_par] = other_leaf_node
        new_key = tuple(dual_leaf_path)
        if new_key not in dual_path_scores:
            dual_path_scores[new_key] = np.array(path_scores[k])
        else:
            dual_path_scores[new_key] += path_scores[k]
    dual_path_scores_all_good_runs.append(dual_path_scores)

In [ ]:
def get_predictions(c_HS, c_SO, c_proximity=1):
    coeffs = np.array([c_HS, c_SO, c_proximity])
    num_correct_pred = 0

    for i, run in enumerate(golden_seeds):
        
        dual_path_scores = dict(dual_path_scores_all_good_runs[i])
        absargmax = tuple(np.abs(golden_weights[i,1,:,0,0:15]).argmax(axis=-1).T)    
        for k in dual_path_scores.keys():
            dual_path_scores[k] = dual_path_scores[k]@coeffs
        max_path_score = max(dual_path_scores.values())

        good_dual_paths = {k for k,v in dual_path_scores.items() if v>= 0.9999999*max_path_score}
        good_dual_paths_no_routes = {key[0:4] for key in good_dual_paths}
        if absargmax in good_dual_paths_no_routes:
            num_correct_pred += 1/len(good_dual_paths_no_routes)
    
    return num_correct_pred


#### Prediction

In [ ]:
#The c's are from the paper
c_HS = 0.4
c_SO = 0.7 
c_proximity = 1. #This is always set to 1.

num_correct_pred = get_predictions(c_HS, c_SO, c_proximity)
print(num_correct_pred)
score = round(num_correct_pred/ len(golden_seeds), 2) * 100
print(score)